In [ ]:
from pathlib import Path
import subprocess
import sys

root = Path.cwd()
if not (root / 'NARCBench_Attack_CoT_Monitor').is_dir():
    root = root / 'CompJailBench'
    if not root.exists():
        subprocess.run(['git', 'clone', 'https://github.com/BohuslavO/CompJailBench.git', str(root)], check=True)
subprocess.run(['nvidia-smi'], check=True)
runner = root / 'NARCBench_Attack_CoT_Monitor/scripts/run_qwen_scoring_batches.py'
assert runner.is_file(), 'Open the published Qwen replay revision of CompJailBench'


In [ ]:
from importlib.metadata import version

requirements = root / 'NARCBench_Attack_CoT_Monitor/requirements-probing-kaggle.txt'
replay_requirements = root / 'qwen-replay-requirements.txt'
replay_requirements.write_text(requirements.read_text().replace('inspect_ai==0.3.251', 'inspect-ai==0.3.260'))
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-r', str(replay_requirements),
    'inspect-evals==0.18.0', 'langgraph==1.1.9', 'gptqmodel==7.10.0',
    'transformers==5.22.2', 'torch==2.15.0', 'agent-client-protocol<1.0', 'python-dotenv',
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'torchaudio'], check=True)
print({name: version(name) for name in ('inspect-ai', 'torch', 'transformers', 'gptqmodel')})


In [ ]:
import os

output = root.parent / 'qwen-monitor-scores'
output.mkdir(parents=True, exist_ok=True)
available_gib = os.sysconf('SC_AVPHYS_PAGES') * os.sysconf('SC_PAGE_SIZE') / 1024**3
cpu_memory = min(24, int(available_gib - 3))
assert cpu_memory >= 7, f'Insufficient system RAM: {available_gib:.1f} GiB available'
gpu_rows = subprocess.check_output([
    'nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits',
], text=True).splitlines()
gpu_memory = min(42, min(int(row) for row in gpu_rows) / 1024 - 3)
assert gpu_memory >= 12, 'Insufficient GPU memory'
command = [
    sys.executable, '-u', str(runner), '--output', str(output),
    '--gpu-memory-gib', str(round(gpu_memory, 1)), '--cpu-memory-gib', str(cpu_memory),
]
print('Persistent result directory', output)
subprocess.run(command + ['--validate-only'], cwd=root, check=True)


In [ ]:
subprocess.run(command + ['--batches', 'smoke'], cwd=root, check=True)


In [ ]:
subprocess.run(command, cwd=root, check=True)


In [ ]:
from IPython.display import FileLink, display

for path in sorted(output.glob('qwen-r0-agentharm-batch-*-scores-*.zip')):
    display(FileLink(str(path)))
